# Cắt clip (bước 03) theo từng part trên Kaggle
Thay cho `data_pipeline/steps/03_cut.py` khi máy local chậm. Dùng **đúng code và `configs/data.yaml`** của local: Silero VAD + YuNet, clip 5–8 s, 25 fps CFR, cạnh ngắn ≤1080.
Đầu vào: thư mục raw của part (kết quả bước 02) đã upload thành Kaggle dataset, gồm `videos/`, `sources.jsonl`, `logs/download_results.csv`.
Bật Internet (cài thư viện, tải YuNet/Silero). Không cần GPU: VAD và dò mặt chạy CPU.
Output: `clips/` + `review.csv` + `logs/`, nén ZIP để tải về local rồi chạy `04_review.py`.

In [ ]:
from pathlib import Path
import subprocess, sys, os
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
REPO_URL = "https://github.com/linhxm/vn-av-forensics.git"  # Sửa nếu đổi repository.
ROOT = Path("/kaggle/working/vn-av-df-forensics")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
PIPELINE = ROOT / "data_pipeline"
assert (PIPELINE / "src/vn_av_data/data/curation.py").is_file(), "Cần push code mới trước khi dùng notebook"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PIPELINE)], check=True)
sys.path.insert(0, str(PIPELINE / "src"))
os.chdir(PIPELINE)  # Đường dẫn model trong configs/data.yaml tính từ data_pipeline/.
from vn_av_data.cli import main

## 1. Chọn part và đường dẫn
`RAW_DATASET` là thư mục chứa `sources.jsonl` (đường dẫn video trong manifest tính tương đối từ file này, nên giữ nguyên cấu trúc khi upload).
Kaggle giới hạn thời gian phiên; cắt ghi nhật ký theo từng video nên chạy lại là cắt tiếp. Sang phiên mới: attach output (hoặc ZIP) của phiên trước rồi đặt `PREVIOUS_CUT`.
Luật cắt, code cắt và model bị khóa bằng chữ ký trong `logs/cut-run.json`: phiên sau và local phải dùng cùng code/config thì mới cắt tiếp được.

In [ ]:
PART = 1  # Ví dụ: 2 -> xử lý vn-av-df-data-part2.
DATASET_VERSION = f"vn-av-df-data-part{PART}"
RAW_DATASET = Path("/kaggle/input/YOUR_RAW_PART") / DATASET_VERSION  # Thư mục có sources.jsonl.
OUTPUT = Path("/kaggle/working/candidates") / DATASET_VERSION
PREVIOUS_CUT = None  # Cắt tiếp phiên trước, ví dụ Path("/kaggle/input/YOUR_CUT_PART") / DATASET_VERSION.
CONFIG = PIPELINE / "configs/data.yaml"  # Giữ như local để clip cùng luật.

manifest = RAW_DATASET / "sources.jsonl"
assert manifest.is_file(), f"Không thấy {manifest}; kiểm tra RAW_DATASET"
if PREVIOUS_CUT is not None and not OUTPUT.exists():
    import shutil
    shutil.copytree(PREVIOUS_CUT, OUTPUT)
    print("Cắt tiếp từ:", PREVIOUS_CUT)
print("Part:", DATASET_VERSION, "| Input:", manifest, "| Output:", OUTPUT)
print(CONFIG.read_text(encoding="utf-8"))

## W&B (tuỳ chọn)
Ghi cấu hình, thống kê và kết quả lên Weights & Biases sau khi chạy xong (không đổi code pipeline).
Lần đầu: thêm Kaggle secret tên `WANDB_API_KEY` (Add-ons → Secrets, bật cho notebook). Đặt `USE_WANDB = False` để tắt.

In [ ]:
import json
USE_WANDB = True
WANDB_PROJECT = "vn-av-df"
WANDB_ENTITY = None  # Team W&B; None = tài khoản của API key.
LOG_SAMPLE_VIDEOS = 4  # Số video mẫu đưa lên W&B; 0 = không.
if USE_WANDB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb
    try:
        from kaggle_secrets import UserSecretsClient
        wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    except Exception as exc:  # Không có secret: tắt W&B, pipeline vẫn chạy.
        USE_WANDB = False
        print("Tắt W&B, không đọc được secret WANDB_API_KEY:", exc)

def plain(value):
    """Cấu hình/metric về kiểu JSON đơn giản để W&B nhận."""
    return json.loads(json.dumps(value, default=str))

## 2. Tải model và cắt
Lỗi ở một video không dừng cả part: video lỗi ghi vào `logs/errors.json`, cuối lượt báo lỗi; chạy lại cell để thử lại các video đó.
Dù cell cắt báo lỗi, vẫn chạy cell ZIP để giữ phần đã cắt.

In [ ]:
main(["setup", "--config", str(CONFIG)])

In [ ]:
main(["cut", "--config", str(CONFIG), "--manifest", str(manifest), "--output", str(OUTPUT)])

In [ ]:
# Ghi kết quả cắt: thống kê, bảng clip, bảng theo video nguồn, lỗi và vài clip mẫu.
# Chạy được cả khi cell cắt báo lỗi (ghi phần đã cắt).
if USE_WANDB:
    import pandas as pd
    logs = OUTPUT / "logs"

    def load(name, default):
        path = logs / name
        return json.loads(path.read_text(encoding="utf-8")) if path.exists() else default

    run = wandb.init(
        project=WANDB_PROJECT, entity=WANDB_ENTITY, job_type="cut", group=DATASET_VERSION,
        name=f"cut-{DATASET_VERSION}",
        config=plain({"part": DATASET_VERSION, **load("cut-run.json", {}).get("config", {})}),
    )
    summary = load("summary.json", {})
    run.summary.update({k: v for k, v in summary.items() if isinstance(v, (int, float))})
    review = OUTPUT / "review.csv"
    clips = pd.read_csv(review, encoding="utf-8-sig") if review.exists() else pd.DataFrame()
    per_source = []
    for sid, done in load("sources.json", {}).items():
        per_source.append({
            "source_id": sid,
            "speaker_id": next((c.get("speaker_id") for c in done["candidates"]), ""),
            "clips": len(done["candidates"]),
            "clip_seconds": round(sum(c["duration_s"] for c in done["candidates"]), 1),
            "rejected_windows": len(done["rejected"]),
            "source_rejected": done.get("source_rejected", ""),
        })
    tables = {"clips": clips, "per_source": pd.DataFrame(per_source),
              "errors": pd.DataFrame(load("errors.json", []))}
    run.log({name: wandb.Table(dataframe=df.astype(str)) for name, df in tables.items() if len(df)})
    if len(clips):
        run.summary.update({"clip_hours": round(float(clips["duration_s"].sum()) / 3600, 3),
                            "speakers": int(clips["speaker_id"].nunique())})
        run.log({"clip_duration_s": wandb.Histogram(clips["duration_s"]),
                 "face_ratio": wandb.Histogram(clips["face_ratio"])})
        # Mỗi video nguồn lấy clip đầu tiên, tối đa LOG_SAMPLE_VIDEOS.
        sample = clips.groupby("source_id").head(1).head(LOG_SAMPLE_VIDEOS)
        if len(sample):
            run.log({"samples": [wandb.Video(str(OUTPUT / p), caption=c)
                                 for p, c in zip(sample["file_path"], sample["clip_id"])]})
    run.finish()

In [ ]:
import json, shutil
logs = OUTPUT / "logs"
for name in ("summary.json", "errors.json"):
    if (logs / name).exists():
        print(name, json.dumps(json.loads((logs / name).read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
archive = shutil.make_archive("/kaggle/working/" + OUTPUT.name, "zip", OUTPUT.parent, OUTPUT.name)
print("Tải part:", archive)

## 3. Review ở local
Giải nén ZIP vào `data_pipeline/data/candidates/vn-av-df-data/`, được thư mục `vn-av-df-data-partN` (có `clips/`, `review.csv`, `logs/`).
Đặt `PART=N` trong `data_settings.py` rồi chạy `data_pipeline/steps/04_review.py` → `05_export.py`.
Tải thêm video cho part: có thể cắt phần mới ở local bằng `03_cut.py` (cùng code/config) hoặc upload lại raw và đặt `PREVIOUS_CUT` trên Kaggle.